# 10.9 · Vision Transformer (ViT)

> **课程定位 / Where this fits**
> 第 9 课，**Part 10 · 计算机视觉**。
> Lesson 9, **Part 10 · Computer Vision**.
>
> 十年来 CNN 统治图像。2020 年 **Vision Transformer(ViT)** 带来全新范式：**把图像切成小块(patch)当作"词"，直接用 NLP 里的 Transformer 自注意力处理**——几乎不用卷积。ViT 在大数据上超越 CNN，是现代多模态/大模型的视觉基石。本课讲清 **patch embedding、class token、位置编码、自注意力**，**从零搭一个迷你 ViT** 训练，**可视化注意力**，并诚实对比 ViT 与 CNN 的取舍。
> CNNs dominated images for a decade. In 2020 the **Vision Transformer (ViT)** brought a new paradigm: **cut the image into patches treated as "words" and process them with NLP's Transformer self-attention** — almost no convolution. ViT beats CNNs on large data and underpins modern multimodal/large models. We'll cover **patch embedding, class token, positional encoding, self-attention**, **build a mini ViT from scratch**, **visualize attention**, and honestly compare ViT vs CNN.
>
> 💼 **实战/面试视角**："ViT 怎么把图变成序列 / 自注意力 / 为什么ViT需要大数据 / CNN vs ViT" 是热门题。
> 💼 **Practical/interview angle:** "how ViT turns an image into a sequence / self-attention / why ViT needs big data / CNN vs ViT" — hot topics.

> 📐 **符号约定 / Notation**
> - patch —— 图像切出的小方块(如 7×7) / a small image square (e.g. 7×7)
> - token —— 一个 patch 经线性投影后的向量 / a patch projected to a vector
> - $Q,K,V$ —— 注意力的 query/key/value / attention query/key/value

> 💡 **面试相关 / Interview-relevant**
> - "ViT 如何把图像变成 Transformer 的输入"（出镜率 ★★★★★，patch+投影+pos）
> - "自注意力 QKV 的计算"（★★★★★）
> - "class token 和位置编码的作用"（★★★★）
> - "为什么 ViT 比 CNN 更吃数据(缺少归纳偏置)"（★★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 ViT 的核心：图像→patch序列→Transformer。
   Understand ViT's core: image → patch sequence → Transformer.
2. 掌握 **patch embedding、class token、位置编码**。
   Master patch embedding, class token, positional encoding.
3. 理解**自注意力**并从零实现一次。
   Understand self-attention and implement it once from scratch.
4. **搭建迷你 ViT** 训练，并**可视化注意力**。
   Build a mini ViT, train it, and visualize attention.
5. 诚实对比 ViT vs CNN（归纳偏置与数据量）。
   Honestly compare ViT vs CNN (inductive bias and data).

## 目录 / TOC
1. [从 CNN 到 ViT：图像即序列 ⭐](#1)
2. [Patch Embedding + class token + 位置编码 ⭐](#2)
3. [自注意力（从零）⭐](#3)
4. [搭建并训练迷你 ViT + 注意力可视化 ⭐](#4)
5. [ViT vs CNN：归纳偏置与数据 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 从 CNN 到 ViT：图像即序列 ⭐ / From CNN to ViT: Image as a Sequence

CNN 靠卷积的**局部性、平移不变**等"归纳偏置(inductive bias)"高效处理图像。**Transformer**(NLP 里处理句子的架构)则靠**自注意力**——让序列里**每个元素与所有其它元素直接交互**，从第一层就有**全局视野**。
CNNs use convolution's "inductive biases" (locality, translation invariance) to process images efficiently. **Transformers** (the NLP architecture for sentences) use **self-attention** — letting **every element interact directly with all others**, giving a **global view from layer 1**.

ViT 的关键一步：**把图像变成一串"词"**。具体做法：把图切成不重叠的小方块(**patch**，如 7×7)，每个 patch 拉平并线性投影成一个向量(**token**)。于是一张图就变成了一个 token 序列——和一句话的词序列一模一样，可以直接喂进 Transformer。
ViT's key move: **turn the image into a sequence of "words."** Cut the image into non-overlapping squares (**patches**, e.g. 7×7); flatten and linearly project each patch into a vector (**token**). Now an image is a token sequence — just like a sentence's words — ready for a Transformer.

**核心区别(面试)**：CNN **自带**局部性等先验，小数据也能学好；ViT **几乎没有图像先验**，全靠数据学习这些结构，所以**需要更多数据(或预训练)**才能发挥，但上限更高、更灵活。
**Key difference (interview):** CNNs have **built-in** priors (locality), learning well on small data; ViTs have **almost no image priors** and must learn structure from data, so they **need more data (or pretraining)** but have a higher ceiling and more flexibility.


<a id="2"></a>
## 2. Patch Embedding + class token + 位置编码 ⭐ / Patch Embedding + Class Token + Positional Encoding

三个组件把图像变成 Transformer 的输入：
Three components turn an image into Transformer input:
1. **Patch embedding**：切 patch → 拉平 → 线性投影成 $d$ 维向量。实现上可以用一个 **kernel=stride=patch 大小的卷积**一步搞定（每个 patch 投影成一个向量）。
   **Patch embedding:** cut patches → flatten → linear-project to $d$-dim. Implemented neatly as **one conv with kernel=stride=patch size** (each patch → one vector).
2. **class token([CLS])**：额外加一个**可学习的 token** 拼在序列最前面。经过 Transformer 后，**用它的输出代表整张图**做分类（它"汇总"了所有 patch 的信息）。
   **Class token ([CLS]):** prepend an extra **learnable token**. After the Transformer, **its output represents the whole image** for classification (it aggregates all patches).
3. **位置编码(positional encoding)**：自注意力本身**不知道 patch 的顺序/位置**（它对输入是"无序"的）。所以要给每个 token 加一个**可学习的位置向量**，告诉模型"这个 patch 在图的哪儿"。
   **Positional encoding:** self-attention is **order-agnostic** (it doesn't know patch positions). So add a **learnable position vector** to each token telling the model "where this patch is."

下面可视化 patchify，并实现 patch embedding。
Let's visualize patchifying and implement patch embedding.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="white")

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.ToTensor()
train_full = torchvision.datasets.FashionMNIST(DATA_ROOT, train=True, download=True, transform=tfm)
test_full  = torchvision.datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=tfm)
classes = train_full.classes

# 可视化: 把一张 28×28 图切成 7×7 的 patch (共 16 个) / visualize patchifying into 4x4=16 patches
img, lab = train_full[0]; P = 7; n = 28 // P
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img.squeeze(), cmap="gray"); axes[0].set_title(f"原图 28×28 ({classes[lab]})")
for i in range(1, n): axes[0].axhline(i*P-0.5, color="r", lw=1); axes[0].axvline(i*P-0.5, color="r", lw=1)
axes[0].axis("off")
# 把每个 patch 单独画出来 / draw each patch separately
grid = np.ones((28+ n-1, 28+ n-1)) * 0.5
patches_img = img.squeeze().unfold(0, P, P).unfold(1, P, P)   # (n,n,P,P) 切块 / unfold into patches
canvas = np.ones((n*(P+1), n*(P+1)))
for r in range(n):
    for c in range(n):
        canvas[r*(P+1):r*(P+1)+P, c*(P+1):c*(P+1)+P] = patches_img[r,c]
axes[1].imshow(canvas, cmap="gray"); axes[1].set_title(f"切成 {n}×{n}={n*n} 个 {P}×{P} patch"); axes[1].axis("off")
plt.tight_layout(); plt.show()
print(f"一张 28×28 图 → {n*n} 个 {P}×{P} 的 patch → 每个 patch 投影成一个向量(token) → 序列长度 {n*n}")


In [ ]:
class PatchEmbed(nn.Module):
    """切 patch 并线性投影: 用 kernel=stride=patch 的卷积一步实现 / patch embedding via strided conv."""
    def __init__(self, img=28, patch=7, dim=64, cin=1):
        super().__init__()
        self.n = (img // patch) ** 2                       # patch 个数 / number of patches
        self.proj = nn.Conv2d(cin, dim, kernel_size=patch, stride=patch)   # 每个patch→dim维向量 / each patch→vector
    def forward(self, x):
        x = self.proj(x)                                   # (B, dim, n', n')
        return x.flatten(2).transpose(1, 2)                # (B, n_patches, dim) 序列 / sequence of tokens

pe = PatchEmbed()
out = pe(img.unsqueeze(0))
print(f"PatchEmbed: 输入 (1,1,28,28) → token 序列 {tuple(out.shape)}  = (batch, 16个patch, 64维)")
print("class token: 再在最前面拼1个可学习token → 序列变 17; 用它的输出代表整图分类")
print("位置编码: 给17个token各加一个可学习位置向量(自注意力本身不知道顺序)")


<a id="3"></a>
## 3. 自注意力（从零）⭐ / Self-Attention (From Scratch)

**自注意力(self-attention)** 是 Transformer 的心脏。直觉：序列里每个 token 都问"**我该关注其它哪些 token？**"，然后按关注度加权汇总它们的信息。
**Self-attention** is the heart of the Transformer. Intuition: each token asks "**which other tokens should I attend to?**" then aggregates their info weighted by attention.

机制(面试必会)：每个 token 生成三个向量——**Query(查询 Q)、Key(键 K)、Value(值 V)**：
Mechanism (must-know): each token produces three vectors — **Query (Q), Key (K), Value (V)**:
- 用 $Q$ 和所有 $K$ 做点积 → 得到"该 token 对其它每个 token 的相关性分数"。
  Dot $Q$ with all $K$ → "relevance scores of this token to every other."
- 除以 $\sqrt{d}$ 缩放后 softmax → 变成**注意力权重**（和为 1）。
  Scale by $\sqrt{d}$, softmax → **attention weights** (sum to 1).
- 用这些权重对所有 $V$ 加权求和 → 该 token 的新表示（融合了相关 token 的信息）。
  Weighted-sum all $V$ → the token's new representation (fused from relevant tokens).

公式：$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$。下面从零实现一次。
Formula: $\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$. Let's implement it from scratch.


In [ ]:
def self_attention(X, Wq, Wk, Wv):
    """从零的单头自注意力 / single-head self-attention from scratch. X:(seq, dim)."""
    Q = X @ Wq; K = X @ Wk; V = X @ Wv                    # 投影出 Q,K,V / project to Q,K,V
    d = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d)                         # 相关性分数 QK^T/√d / scaled scores
    attn = torch.softmax(scores, dim=-1)                  # softmax → 注意力权重(每行和为1) / attention weights
    out = attn @ V                                        # 按权重汇总 V / weighted sum of V
    return out, attn

torch.manual_seed(0)
seq, dim = 5, 8                                           # 5 个 token, 8 维 / 5 tokens, dim 8
X = torch.randn(seq, dim)
Wq, Wk, Wv = torch.randn(dim,dim), torch.randn(dim,dim), torch.randn(dim,dim)
out, attn = self_attention(X, Wq, Wk, Wv)
print(f"输入 {tuple(X.shape)} → 输出 {tuple(out.shape)}; 注意力矩阵 {tuple(attn.shape)} (每行=一个token对其它token的关注度)")
print(f"注意力权重每行之和 = {attn.sum(1).round(decimals=3).tolist()} (softmax 保证和为1)")
fig, ax = plt.subplots(figsize=(4.5,3.8)); im = ax.imshow(attn, cmap="viridis")
ax.set_xlabel("被关注的 token (Key)"); ax.set_ylabel("发起关注的 token (Query)")
ax.set_title("自注意力矩阵: 每个token关注谁"); plt.colorbar(im); plt.tight_layout(); plt.show()
print("自注意力: 每token用Q与所有K点积得分→softmax→加权汇总V; 全局交互, 这是Transformer的核心")
print("多头注意力(multi-head): 并行做多组Q/K/V, 关注不同方面再拼接")


<a id="4"></a>
## 4. 搭建并训练迷你 ViT + 注意力可视化 ⭐ / Build & Train a Mini ViT

把组件拼起来：**patch embed → 加 class token + 位置编码 → 几层 Transformer 块 → 取 class token 输出分类**。用 PyTorch 的 `nn.MultiheadAttention` 搭 Transformer 块（同时能取出注意力权重做可视化），在 FashionMNIST 上训练。
Assemble it: **patch embed → add class token + positional encoding → a few Transformer blocks → classify from the class token**. We use PyTorch's `nn.MultiheadAttention` (which also exposes attention weights for visualization) and train on FashionMNIST.


In [ ]:
class Block(nn.Module):
    """一个 Transformer 块: 注意力 + MLP, 都带残差(呼应10.4) / a Transformer block."""
    def __init__(self, dim, heads=4, mlp=128):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp), nn.GELU(), nn.Linear(mlp, dim))
        self.last_attn = None
    def forward(self, x):
        h = self.n1(x)
        a, w = self.attn(h, h, h, need_weights=True, average_attn_weights=True)  # 自注意力+取权重 / attn + weights
        self.last_attn = w.detach()                        # 存注意力权重供可视化 / save for viz
        x = x + a                                          # 残差 / residual
        x = x + self.mlp(self.n2(x))                       # 残差 MLP / residual MLP
        return x

class MiniViT(nn.Module):
    def __init__(self, img=28, patch=7, dim=64, depth=2, heads=4, n_cls=10):
        super().__init__()
        self.embed = PatchEmbed(img, patch, dim)
        n = self.embed.n
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))            # 可学习 class token / learnable [CLS]
        self.pos = nn.Parameter(torch.randn(1, n+1, dim) * 0.02)   # 位置编码(含cls) / positional encoding
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim); self.head = nn.Linear(dim, n_cls)
    def forward(self, x):
        B = x.shape[0]
        x = self.embed(x)                                          # (B, n, dim) patch tokens
        cls = self.cls.expand(B, -1, -1)                          # 复制 cls 到每个样本 / broadcast cls
        x = torch.cat([cls, x], dim=1) + self.pos                 # 拼cls + 加位置编码 / prepend cls + add pos
        for blk in self.blocks: x = blk(x)
        return self.head(self.norm(x[:, 0]))                      # 取 class token(第0个)分类 / classify from [CLS]

train_loader = DataLoader(Subset(train_full, range(12000)), batch_size=128, shuffle=True)
test_loader  = DataLoader(Subset(test_full, range(2000)), batch_size=256)
torch.manual_seed(0); vit = MiniViT(); opt = torch.optim.Adam(vit.parameters(), lr=3e-4); ce = nn.CrossEntropyLoss()
for ep in range(6):
    vit.train()
    for xb, yb in train_loader: opt.zero_grad(); ce(vit(xb), yb).backward(); opt.step()
vit.eval(); correct=total=0
with torch.no_grad():
    for xb,yb in test_loader: correct+=(vit(xb).argmax(1)==yb).sum().item(); total+=len(yb)
acc_vit = correct/total
print(f"迷你 ViT: 参数 {sum(p.numel() for p in vit.parameters()):,}, test 准确率 = {acc_vit:.3f}")


In [ ]:
# 可视化 class token 对各 patch 的注意力 / visualize [CLS] attention over patches
sample, lab = test_full[1]
with torch.no_grad(): _ = vit(sample.unsqueeze(0))
attn = vit.blocks[-1].last_attn[0]                          # 最后一层注意力 (17×17) / last-layer attention
cls_attn = attn[0, 1:]                                      # class token(第0行) 对 16 个 patch 的关注 / CLS→patches
n = int(len(cls_attn) ** 0.5)
amap = cls_attn.reshape(n, n).numpy()
amap_up = np.kron(amap, np.ones((7, 7)))                    # 放大回 28×28 便于叠加 / upscale to image size
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
axes[0].imshow(sample.squeeze(), cmap="gray"); axes[0].set_title(f"输入: {classes[lab]}"); axes[0].axis("off")
axes[1].imshow(amap, cmap="hot"); axes[1].set_title("class token 对4×4 patch 的注意力"); axes[1].axis("off")
axes[2].imshow(sample.squeeze(), cmap="gray"); axes[2].imshow(amap_up, cmap="hot", alpha=0.5)
axes[2].set_title("注意力叠加(亮=更关注)"); axes[2].axis("off")
plt.tight_layout(); plt.show()
print("class token 学会更关注与类别相关的 patch(物体所在区域); 这是 ViT 可解释性的一个窗口")


<a id="5"></a>
## 5. ViT vs CNN：归纳偏置与数据 + 小结 ⭐ / ViT vs CNN: Inductive Bias & Data

在**同样的小数据**(FashionMNIST 子集)上，把迷你 ViT 和参数相近的小 CNN 对比。
On the **same small data** (FashionMNIST subset), compare the mini ViT to a small CNN of similar size.


In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32*7*7,10))
torch.manual_seed(0); opt = torch.optim.Adam(cnn.parameters(), 1e-3)
for ep in range(6):
    cnn.train()
    for xb,yb in train_loader: opt.zero_grad(); ce(cnn(xb),yb).backward(); opt.step()
cnn.eval(); correct=total=0
with torch.no_grad():
    for xb,yb in test_loader: correct+=(cnn(xb).argmax(1)==yb).sum().item(); total+=len(yb)
acc_cnn = correct/total
print(f"小 CNN:   参数 {sum(p.numel() for p in cnn.parameters()):,}, test 准确率 = {acc_cnn:.3f}")
print(f"迷你 ViT: test 准确率 = {acc_vit:.3f}")
print(f"\n在这点小数据上, CNN {'领先' if acc_cnn>acc_vit else '不及'} ViT {abs(acc_cnn-acc_vit):.3f}")
print("诚实结论(面试核心): CNN 自带局部性等归纳偏置, 小数据更省心更强;")
print("ViT 几乎无图像先验, 需大数据/预训练才能反超 → 实战中 ViT 多用 ImageNet 预训练权重再微调")


```
ViT 范式: 图像切patch→线性投影成token序列→Transformer自注意力处理(几乎无卷积)
三组件: patch embedding(切块+投影) + class token(代表整图分类) + 位置编码(告诉顺序)
自注意力: 每token生成Q/K/V; softmax(QK^T/√d)V; 全局交互; 多头=并行多组关注不同方面
class token 注意力可视化: 学会关注与类别相关的区域
CNN vs ViT: CNN有归纳偏置(局部/平移不变)小数据强; ViT无先验需大数据/预训练, 上限更高更灵活
实战: ViT 通常用大规模预训练(ImageNet/JFT)再微调; 小数据优先 CNN 或预训练 ViT
```

### 💡 面试速查 / Interview cheat-sheet
1. **图像→序列**: 切 patch → 投影成 token + class token + 位置编码。
   Image→sequence: patches → token projection + class token + positional encoding.
2. **自注意力**: softmax(QK^T/√d)V, 每token与所有token全局交互。
   Self-attention: softmax(QK^T/√d)V; global all-to-all interaction.
3. **class token**: 汇总全图信息, 用其输出分类。
   Class token: aggregates the image, used for classification.
4. **位置编码**: 自注意力无序, 需显式加位置信息。
   Positional encoding: attention is order-agnostic, add positions explicitly.
5. **ViT vs CNN**: ViT 无归纳偏置, 更吃数据(需预训练); CNN 小数据更稳。
   ViT vs CNN: ViT lacks inductive bias, data-hungry (needs pretraining); CNN better on small data.

### 下一节 / Next
**10.10 多模态 (CLIP)**——让模型同时理解**图像和文字**。CLIP 用对比学习把图片和它的文字描述"对齐"到同一空间，从而实现惊人的**零样本分类**。我们会用合成图文对演示对比学习的核心思想。
**10.10 Multimodal (CLIP)** — models that understand **images and text together**. CLIP uses contrastive learning to align images with their captions in one space, enabling striking **zero-shot classification**. We'll demo the core contrastive idea with synthetic image-text pairs.
